# QLoRA for EGAIS assistant (Colab T4)

This notebook trains a LoRA adapter from `data/lora/train.jsonl` generated by `scripts/build_lora_dataset.py`.

In [ ]:
!pip install -q -U pip
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Put train/eval files to Drive and update paths:
TRAIN_PATH = '/content/drive/MyDrive/egais-lora/train.jsonl'
EVAL_PATH = '/content/drive/MyDrive/egais-lora/eval.jsonl'
OUT_DIR = '/content/drive/MyDrive/egais-lora/adapter-qwen25-15b'

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={'train': TRAIN_PATH, 'eval': EVAL_PATH})
dataset

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN = 1536
LR = 2e-4
EPOCHS = 2
BATCH = 2
GRAD_ACC = 8

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

def format_example(ex):
    msgs = ex['messages']
    text = ''
    for m in msgs:
        text += f"<{m['role']}>\n{m['content']}\n"
    return {'text': text}

train_ds = dataset['train'].map(format_example)
eval_ds = dataset['eval'].map(format_example) if 'eval' in dataset else None

args = TrainingArguments(
    output_dir='/content/out',
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=20,
    save_strategy='epoch',
    eval_strategy='epoch' if eval_ds is not None else 'no',
    bf16=False,
    fp16=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    peft_config=peft_config,
    args=args,
)

trainer.train()
trainer.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print('Saved adapter to', OUT_DIR)

In [ ]:
# Quick sanity check in Colab
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map='auto', torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base, OUT_DIR)
tok = AutoTokenizer.from_pretrained(BASE_MODEL)

prompt = 'Вопрос: какие документы нужны для лицензии на перевозку этилового спирта?\nОтвет:'
inputs = tok(prompt, return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))